# YouTube Sentiment Analysis & AI-Assisted Content Recommendation System
**Ready-to-run Google Colab project**

This notebook collects publicly available YouTube video metadata and comments for a brand/query, cleans the data, performs multilingual sentiment analysis, compares content categories and engagement, creates charts, and exports CSV/Excel/report files.

## Before you run
1. In Google Colab, open **Secrets (🔑)** and add `YOUTUBE_API_KEY`.
2. Enable notebook access to that secret.
3. Run the cells in order.
4. Enter the brand name or channel/video search phrase when prompted.

**Important limitations:** The YouTube API may not expose every comment; some videos have comments disabled. The number collected depends on API quota and comment availability. Sentiment models can misclassify sarcasm, Roman Urdu, slang, and mixed-language comments. Treat outputs as estimates and validate a sample manually. Do not publish personal information from comments.

## 1. Install dependencies

In [ ]:
!pip -q install google-api-python-client pandas matplotlib seaborn openpyxl transformers torch tqdm

## 2. Imports and configuration

In [ ]:
import os, re, json, time, warnings
from collections import Counter
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from google.colab import userdata

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)

# Edit these settings if needed.
MAX_VIDEOS = 40             # Search up to this many videos
MAX_COMMENTS_PER_VIDEO = 250  # API quota permitting
TARGET_COMMENTS = 2000      # Aim only; actual collection may be lower
OUTPUT_DIR = "/content/youtube_sentiment_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

brand_query = input("Enter Pakistani brand or YouTube search phrase: ").strip()
if not brand_query:
    raise ValueError("Please enter a brand name or search phrase.")

API_KEY = userdata.get("YOUTUBE_API_KEY")
if not API_KEY:
    raise ValueError("Missing YOUTUBE_API_KEY. Add it in Colab Secrets and enable notebook access.")

youtube = build("youtube", "v3", developerKey=API_KEY)
print("API client initialized for query:", brand_query)

## 3. Find relevant videos
This searches YouTube for videos related to the brand/query. Review the resulting list to ensure the videos actually belong to the intended brand. You can remove irrelevant video IDs from the resulting dataframe before collecting comments.

In [ ]:
def search_videos(query, max_videos=40):
    results = []
    page_token = None

    while len(results) < max_videos:
        request = youtube.search().list(
            part="snippet",
            q=query,
            type="video",
            order="relevance",
            maxResults=min(50, max_videos - len(results)),
            pageToken=page_token
        )
        response = request.execute()

        for item in response.get("items", []):
            sn = item["snippet"]
            results.append({
                "video_id": item["id"]["videoId"],
                "video_title": sn.get("title", ""),
                "channel_title": sn.get("channelTitle", ""),
                "channel_id": sn.get("channelId", ""),
                "published_at": sn.get("publishedAt", ""),
                "description": sn.get("description", "")
            })

        page_token = response.get("nextPageToken")
        if not page_token:
            break

    return pd.DataFrame(results).drop_duplicates("video_id").reset_index(drop=True)

videos_df = search_videos(brand_query, MAX_VIDEOS)
print(f"Videos found: {len(videos_df)}")
display(videos_df[["video_id", "video_title", "channel_title", "published_at"]].head(40))

videos_path = os.path.join(OUTPUT_DIR, "videos_found.csv")
videos_df.to_csv(videos_path, index=False, encoding="utf-8-sig")
print("Saved:", videos_path)

## 4. Collect public comments and engagement statistics
This uses the official YouTube Data API. API quota limits apply. If a video has disabled comments or the API rejects access, the notebook skips it and records the reason.

In [ ]:
def get_video_statistics(video_ids):
    stats = {}
    for start in range(0, len(video_ids), 50):
        batch = video_ids[start:start+50]
        if not batch:
            continue
        response = youtube.videos().list(
            part="statistics,contentDetails",
            id=",".join(batch),
            maxResults=50
        ).execute()
        for item in response.get("items", []):
            st = item.get("statistics", {})
            stats[item["id"]] = {
                "view_count": int(st.get("viewCount", 0)),
                "like_count": int(st.get("likeCount", 0)),
                "comment_count_reported": int(st.get("commentCount", 0)),
                "duration": item.get("contentDetails", {}).get("duration", "")
            }
    return stats

def collect_comments(video_id, video_title, channel_title, max_comments=250):
    rows = []
    page_token = None
    try:
        while len(rows) < max_comments:
            response = youtube.commentThreads().list(
                part="snippet",
                videoId=video_id,
                maxResults=min(100, max_comments - len(rows)),
                pageToken=page_token,
                textFormat="plainText",
                order="relevance"
            ).execute()

            for item in response.get("items", []):
                top = item["snippet"]["topLevelComment"]["snippet"]
                rows.append({
                    "video_id": video_id,
                    "video_title": video_title,
                    "channel_title": channel_title,
                    "comment_id": item["snippet"]["topLevelComment"]["id"],
                    "comment_text": top.get("textDisplay", ""),
                    "author_name": top.get("authorDisplayName", ""),
                    "published_at": top.get("publishedAt", ""),
                    "like_count": int(top.get("likeCount", 0))
                })
                if len(rows) >= max_comments:
                    break

            page_token = response.get("nextPageToken")
            if not page_token:
                break

    except HttpError as e:
        print(f"Skipped {video_id}: {e}")
    return rows

video_ids = videos_df["video_id"].tolist()
video_stats = get_video_statistics(video_ids)

all_comments = []
errors_or_empty = []
for _, v in tqdm(videos_df.iterrows(), total=len(videos_df), desc="Collecting comments"):
    comments = collect_comments(
        v["video_id"], v["video_title"], v["channel_title"],
        MAX_COMMENTS_PER_VIDEO
    )
    if not comments:
        errors_or_empty.append({"video_id": v["video_id"], "video_title": v["video_title"], "note": "No accessible top-level comments returned"})
    all_comments.extend(comments)
    if len(all_comments) >= TARGET_COMMENTS:
        break

comments_df = pd.DataFrame(all_comments)
if comments_df.empty:
    raise RuntimeError("No comments were collected. Check API access, query relevance, and whether comments are enabled.")

stats_df = pd.DataFrame([
    {"video_id": vid, **stat} for vid, stat in video_stats.items()
])
comments_df = comments_df.merge(stats_df, on="video_id", how="left")

print(f"Collected {len(comments_df):,} raw comments from {comments_df['video_id'].nunique()} videos.")
print(f"Videos with no accessible comments: {len(errors_or_empty)}")
display(comments_df.head())

## 5. Clean comments
We remove blank entries, duplicate comment IDs/text, and comments containing no letters or numbers. Emoji-only entries are removed by the same text-content check. Original text is retained for analysis.

In [ ]:
def clean_comment_text(text):
    text = str(text or "")
    text = re.sub(r"\s+", " ", text).strip()
    return text

comments_df["comment_text"] = comments_df["comment_text"].map(clean_comment_text)
comments_df["comment_text_lower"] = comments_df["comment_text"].str.lower()

before = len(comments_df)
clean_df = comments_df[
    comments_df["comment_text"].str.len().gt(0)
    & comments_df["comment_text"].str.contains(r"[A-Za-z0-9\u0600-\u06FF]", regex=True, na=False)
].copy()

clean_df = clean_df.drop_duplicates(subset=["comment_id"])
clean_df = clean_df.drop_duplicates(subset=["video_id", "comment_text_lower"])
clean_df = clean_df.drop(columns=["comment_text_lower"], errors="ignore")
clean_df = clean_df.reset_index(drop=True)

print(f"Rows before cleaning: {before:,}")
print(f"Rows after cleaning:  {len(clean_df):,}")
print(f"Removed:              {before-len(clean_df):,}")

## 6. Sentiment analysis
The default model is a multilingual XLM-RoBERTa sentiment model. It is a zero-shot-use pretrained sentiment classifier, not a model trained specifically on Pakistani brand comments. Roman Urdu performance is uncertain, so manually review a sample before relying on results.

In [ ]:
from transformers import pipeline
import torch

MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"
device = 0 if torch.cuda.is_available() else -1

sentiment_pipe = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME,
    tokenizer=MODEL_NAME,
    device=device
)

def map_sentiment_label(label):
    label = str(label).lower()
    if "positive" in label or label.endswith("2"):
        return "Positive"
    if "negative" in label or label.endswith("0"):
        return "Negative"
    if "neutral" in label or label.endswith("1"):
        return "Neutral"
    return label.title()

texts = clean_df["comment_text"].astype(str).tolist()
predictions = []
BATCH_SIZE = 32

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Scoring sentiment"):
    batch = texts[start:start+BATCH_SIZE]
    batch_results = sentiment_pipe(batch, truncation=True, max_length=256)
    predictions.extend(batch_results)

clean_df["sentiment"] = [map_sentiment_label(p["label"]) for p in predictions]
clean_df["sentiment_confidence"] = [float(p["score"]) for p in predictions]

display(clean_df[["comment_text", "sentiment", "sentiment_confidence"]].head(15))

## 7. Categorize videos by title keywords
This is a transparent keyword-based grouping, not a trained classifier. Add or edit keywords to fit the selected brand and campaign context.

In [ ]:
def categorize_video(title):
    t = str(title).lower()
    rules = {
        "Eid/Festive": ["eid", "ramadan", "ramzan", "iftar", "festive", "independence day", "14 august", "new year"],
        "Sale/Discount": ["sale", "discount", "offer", "deal", "flat off", "clearance", "promo", "promotion"],
        "New Product/Collection": ["new collection", "launch", "launched", "new arrival", "unboxing", "new product", "introducing"],
        "Brand/Other": []
    }
    for category, keywords in rules.items():
        if any(keyword in t for keyword in keywords):
            return category
    return "Brand/Other"

videos_df["content_category"] = videos_df["video_title"].map(categorize_video)
clean_df = clean_df.merge(
    videos_df[["video_id", "content_category"]],
    on="video_id",
    how="left",
    suffixes=("", "_video")
)
clean_df["content_category"] = clean_df["content_category"].fillna("Brand/Other")

# One row per video for engagement comparisons
video_summary = videos_df.copy()
video_summary["content_category"] = video_summary["video_title"].map(categorize_video)
video_summary = video_summary.merge(stats_df, on="video_id", how="left")
video_summary["collected_comments"] = video_summary["video_id"].map(clean_df.groupby("video_id").size()).fillna(0).astype(int)

display(video_summary[["video_title", "content_category", "view_count", "like_count", "comment_count_reported", "collected_comments"]].head(20))

## 8. Analysis tables
Engagement numbers are API-reported counts at collection time. They are not proof that a specific content category caused higher engagement; other factors (audience size, timing, video length, paid promotion) can affect results.

In [ ]:
sentiment_summary = (
    clean_df.groupby(["content_category", "sentiment"])
    .size().unstack(fill_value=0)
)
for col in ["Positive", "Neutral", "Negative"]:
    if col not in sentiment_summary.columns:
        sentiment_summary[col] = 0
sentiment_summary["total_comments"] = sentiment_summary[["Positive", "Neutral", "Negative"]].sum(axis=1)
sentiment_summary["positive_pct"] = (sentiment_summary["Positive"] / sentiment_summary["total_comments"] * 100).round(1)
sentiment_summary["negative_pct"] = (sentiment_summary["Negative"] / sentiment_summary["total_comments"] * 100).round(1)
sentiment_summary = sentiment_summary.sort_values("total_comments", ascending=False)

engagement_summary = (
    video_summary.groupby("content_category", dropna=False)
    .agg(
        videos=("video_id", "nunique"),
        avg_views=("view_count", "mean"),
        avg_likes=("like_count", "mean"),
        avg_reported_comments=("comment_count_reported", "mean"),
        avg_collected_comments=("collected_comments", "mean")
    )
    .round(1)
)

print("Sentiment by content category")
display(sentiment_summary)
print("Engagement by content category")
display(engagement_summary)

## 9. Charts

In [ ]:
# Chart 1: sentiment counts
sentiment_counts = clean_df["sentiment"].value_counts().reindex(["Positive", "Neutral", "Negative"], fill_value=0)
plt.figure(figsize=(7, 4))
sentiment_counts.plot(kind="bar")
plt.title("Overall YouTube Comment Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Number of comments")
plt.xticks(rotation=0)
plt.tight_layout()
chart1 = os.path.join(OUTPUT_DIR, "sentiment_distribution.png")
plt.savefig(chart1, dpi=160)
plt.show()

# Chart 2: average engagement by content category
plot_data = engagement_summary[["avg_likes", "avg_reported_comments"]].copy()
if not plot_data.empty:
    ax = plot_data.plot(kind="bar", figsize=(9, 5))
    ax.set_title("Average Engagement by Video Category")
    ax.set_xlabel("Video category")
    ax.set_ylabel("Average count per video")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    chart2 = os.path.join(OUTPUT_DIR, "engagement_by_category.png")
    plt.savefig(chart2, dpi=160)
    plt.show()

# Chart 3: positive sentiment percentage by category
positive_plot = sentiment_summary["positive_pct"].sort_values(ascending=False)
plt.figure(figsize=(9, 4))
positive_plot.plot(kind="bar")
plt.title("Positive Comment Share by Content Category")
plt.xlabel("Video category")
plt.ylabel("Positive comments (%)")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
chart3 = os.path.join(OUTPUT_DIR, "positive_share_by_category.png")
plt.savefig(chart3, dpi=160)
plt.show()

## 10. Recommendation engine
Recommendations are generated from observed category-level engagement and sentiment. Categories with very few comments should be interpreted cautiously.

In [ ]:
def generate_recommendations(sentiment_table, engagement_table, min_comments=20):
    recommendations = []
    usable = sentiment_table[sentiment_table["total_comments"] >= min_comments].copy()

    if usable.empty:
        recommendations.append(
            f"Collect more comments before making strong content recommendations; no category reached {min_comments} comments."
        )
    else:
        best_positive = usable["positive_pct"].idxmax()
        recommendations.append(
            f"Consider testing more '{best_positive}' content: it has the highest positive-comment share "
            f"({usable.loc[best_positive, 'positive_pct']:.1f}%) among categories with at least {min_comments} comments."
        )

        worst_negative = usable["negative_pct"].idxmax()
        recommendations.append(
            f"Review '{worst_negative}' content for recurring complaints; it has the highest negative-comment share "
            f"({usable.loc[worst_negative, 'negative_pct']:.1f}%) among categories with at least {min_comments} comments."
        )

    if not engagement_table.empty:
        best_likes = engagement_table["avg_likes"].idxmax()
        recommendations.append(
            f"'{best_likes}' has the highest average likes per video in this sample "
            f"({engagement_table.loc[best_likes, 'avg_likes']:.1f}). Test similar formats, but avoid assuming causation."
        )
        best_comments = engagement_table["avg_reported_comments"].idxmax()
        recommendations.append(
            f"'{best_comments}' has the highest average API-reported comments per video "
            f"({engagement_table.loc[best_comments, 'avg_reported_comments']:.1f}). Inspect those videos to identify possible themes."
        )

    return recommendations

recommendations = generate_recommendations(sentiment_summary, engagement_summary)
print("CONTENT RECOMMENDATIONS")
for i, item in enumerate(recommendations, 1):
    print(f"{i}. {item}")

## 11. Export results and generate a report
Files are saved to `/content/youtube_sentiment_output`. At the end, a ZIP is created and offered for download.

In [ ]:
# Add collection metadata
collection_metadata = {
    "brand_query": brand_query,
    "collection_date_utc": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "videos_found": int(len(videos_df)),
    "videos_with_comments": int(clean_df["video_id"].nunique()),
    "comments_after_cleaning": int(len(clean_df)),
    "sentiment_model": MODEL_NAME,
    "notes": [
        "YouTube API results are limited by quota and comment availability.",
        "Sentiment labels are model estimates; manually validate a sample, especially Roman Urdu.",
        "Video categories use title keywords and may be imperfect.",
        "Observed associations do not establish causation."
    ]
}
with open(os.path.join(OUTPUT_DIR, "collection_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(collection_metadata, f, ensure_ascii=False, indent=2)

videos_df.to_csv(os.path.join(OUTPUT_DIR, "videos_found.csv"), index=False, encoding="utf-8-sig")
clean_df.to_csv(os.path.join(OUTPUT_DIR, "cleaned_comments_with_sentiment.csv"), index=False, encoding="utf-8-sig")
sentiment_summary.to_csv(os.path.join(OUTPUT_DIR, "sentiment_summary_by_category.csv"), encoding="utf-8-sig")
engagement_summary.to_csv(os.path.join(OUTPUT_DIR, "engagement_summary_by_category.csv"), encoding="utf-8-sig")
video_summary.to_csv(os.path.join(OUTPUT_DIR, "video_level_summary.csv"), index=False, encoding="utf-8-sig")

# Excel workbook
excel_path = os.path.join(OUTPUT_DIR, "youtube_sentiment_analysis.xlsx")
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    videos_df.to_excel(writer, sheet_name="Videos Found", index=False)
    clean_df.to_excel(writer, sheet_name="Comments Sentiment", index=False)
    sentiment_summary.to_excel(writer, sheet_name="Sentiment Summary")
    engagement_summary.to_excel(writer, sheet_name="Engagement Summary")
    video_summary.to_excel(writer, sheet_name="Video Summary", index=False)

report_lines = [
    f"# YouTube Sentiment Analysis Report: {brand_query}",
    "",
    f"**Collection date (UTC):** {collection_metadata['collection_date_utc']}",
    f"**Videos found:** {len(videos_df)}",
    f"**Videos represented in cleaned comments:** {clean_df['video_id'].nunique()}",
    f"**Cleaned comments analyzed:** {len(clean_df)}",
    f"**Sentiment model:** `{MODEL_NAME}`",
    "",
    "## Methodology",
    "Videos were discovered using the YouTube Data API. Publicly accessible top-level comments were collected subject to API quota and video comment settings. Empty and duplicate comments were removed. A pretrained multilingual sentiment model assigned positive, neutral, or negative labels. Video titles were grouped using keyword rules. Engagement metrics were summarized by content category.",
    "",
    "## Overall sentiment",
    clean_df["sentiment"].value_counts().to_frame("comments").to_markdown(),
    "",
    "## Sentiment by category",
    sentiment_summary.to_markdown(),
    "",
    "## Engagement by category",
    engagement_summary.to_markdown(),
    "",
    "## Recommendations",
]
report_lines += [f"{i}. {r}" for i, r in enumerate(recommendations, 1)]
report_lines += [
    "",
    "## Limitations",
    "- The API may not return every comment, and comments may be disabled on some videos.",
    "- Results reflect the videos returned for the selected search query and may not represent the brand's entire audience.",
    "- Sentiment models can misclassify sarcasm, slang, mixed languages, and Roman Urdu; validate a manually reviewed sample.",
    "- Title keyword categories are approximate and should be reviewed by the researcher.",
    "- Observational results show associations, not causal effects.",
    "- Meta/Facebook/Instagram data was not collected in this project.",
    "",
    "## Reproducibility",
    "The notebook, settings, CSV files, Excel workbook, and charts should be retained with this report. Record any changes to search terms, model, or keyword rules."
]
report_path = os.path.join(OUTPUT_DIR, "final_report.md")
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("Exported files:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

## 12. Download all outputs as one ZIP

In [ ]:
import shutil
zip_path = "/content/youtube_sentiment_results.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)
shutil.make_archive(zip_path.replace(".zip", ""), "zip", OUTPUT_DIR)

from google.colab import files
files.download(zip_path)